In [2]:
import re
import math
import nltk
from collections import defaultdict, Counter
import numpy as np
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Ensure NLTK components are ready
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

True

In [23]:
def func(t,stp=True,lmr=True):
    t=t.lower()
    t=re.sub(r'<[^>]+>','',t)
    t=re.sub(r'[^a-z\s]','',t)
    t=re.sub(r'\s+',' ',t).strip()

    tokens=word_tokenize(t)
    filtered=[]
    if stp:
        stop_words=set(stopwords.words('english'))
        for x in tokens:
            if x not in stop_words:
                filtered.append(x)
    answer=[]            
    if lmr:
        z=WordNetLemmatizer()
        for i in filtered:
            answer.append(z.lemmatize(i))

    return answer

    

corpus = [
    "The quick brown fox jumps over the lazy dog.",
    "The lazy dog barks loudly at the moon.",
    "Foxes are quick and foxes are clever animals."
]
results = []
for c in corpus:
    results.append(func(c))
for res in results:
    print(res)

    



['quick', 'brown', 'fox', 'jump', 'lazy', 'dog']
['lazy', 'dog', 'bark', 'loudly', 'moon']
['fox', 'quick', 'fox', 'clever', 'animal']


# TF-IDF Calculation Example

## Preprocessed Corpus

```python
corpus = [
    "quick brown fox jump lazy dog",
    "lazy dog bark loudly moon",
    "fox quick fox clever animal"
]
```

Total number of documents:

$$
N = 3
$$

---

# Step 1: Bag of Words (BoW)

## Vocabulary

```text
['animal', 'bark', 'brown', 'clever', 'dog',
 'fox', 'jump', 'lazy', 'loudly', 'moon', 'quick']
```

## BoW Matrix

| Document | animal | bark | brown | clever | dog | fox | jump | lazy | loudly | moon | quick |
|-----------|:------:|:----:|:------:|:-------:|:---:|:---:|:----:|:----:|:------:|:----:|:-----:|
| D1 | 0 | 0 | 1 | 0 | 1 | 1 | 1 | 1 | 0 | 0 | 1 |
| D2 | 0 | 1 | 0 | 0 | 1 | 0 | 0 | 1 | 1 | 1 | 0 |
| D3 | 1 | 0 | 0 | 1 | 0 | 2 | 0 | 0 | 0 | 0 | 1 |

---

# Step 2: Document Frequency (DF)

Document Frequency (DF) is the number of documents containing a particular word.

| Term | DF |
|------|---:|
| animal | 1 |
| bark | 1 |
| brown | 1 |
| clever | 1 |
| dog | 2 |
| fox | 2 |
| jump | 1 |
| lazy | 2 |
| loudly | 1 |
| moon | 1 |
| quick | 2 |

---

# Step 3: Calculate IDF

Scikit-learn uses the following formula:

$$
IDF(t)=\ln\left(\frac{1+N}{1+DF(t)}\right)+1
$$

where

- $N = 3$
- $\ln$ denotes the natural logarithm.

### Example 1: Word Appearing in One Document

For **brown**, $DF=1$.

$$
IDF=\ln\left(\frac{1+3}{1+1}\right)+1
$$

$$
=\ln\left(\frac{4}{2}\right)+1
$$

$$
=\ln(2)+1
$$

$$
=0.6931+1
$$

$$
=1.6931
$$

Therefore,

```text
animal
bark
brown
clever
jump
loudly
moon
```

all have

$$
IDF=1.6931
$$

---

### Example 2: Word Appearing in Two Documents

For **fox**, $DF=2$.

$$
IDF=\ln\left(\frac{4}{3}\right)+1
$$

$$
=0.2877+1
$$

$$
=1.2877
$$

Therefore,

```text
dog
fox
lazy
quick
```

all have

$$
IDF=1.2877
$$

---

# Step 4: Raw TF-IDF Calculation

Scikit-learn uses the raw count as Term Frequency (TF).

$$
TF = \text{Raw Count}
$$

and

$$
TF\text{-}IDF = TF \times IDF
$$

---

## Document 1

```text
quick brown fox jump lazy dog
```

### Raw TF-IDF

| Word | TF | IDF | TF × IDF |
|------|---:|----:|---------:|
| quick | 1 | 1.2877 | 1.2877 |
| brown | 1 | 1.6931 | 1.6931 |
| fox | 1 | 1.2877 | 1.2877 |
| jump | 1 | 1.6931 | 1.6931 |
| lazy | 1 | 1.2877 | 1.2877 |
| dog | 1 | 1.2877 | 1.2877 |

### L2 Normalization

$$
Norm=
\sqrt{
1.6931^2+
1.2877^2+
1.2877^2+
1.6931^2+
1.2877^2+
1.2877^2
}
$$

$$
=
\sqrt{
2(2.8666)+4(1.6581)
}
$$

$$
=
\sqrt{12.3656}
$$

$$
=3.5165
$$

Normalized values:

| Word | Value |
|------|------:|
| brown | $1.6931/3.5165 = 0.481$ |
| dog | $1.2877/3.5165 = 0.366$ |
| fox | 0.366 |
| jump | 0.481 |
| lazy | 0.366 |
| quick | 0.366 |

---

## Document 2

```text
lazy dog bark loudly moon
```

### Raw TF-IDF

| Word | TF × IDF |
|------|---------:|
| bark | 1.6931 |
| dog | 1.2877 |
| lazy | 1.2877 |
| loudly | 1.6931 |
| moon | 1.6931 |

### L2 Normalization

$$
Norm=
\sqrt{
3(1.6931^2)+2(1.2877^2)
}
$$

$$
=
\sqrt{
3(2.8666)+2(1.6581)
}
$$

$$
=
\sqrt{11.916}
$$

$$
=3.452
$$

Normalized values:

| Word | Value |
|------|------:|
| bark | 0.490 |
| dog | 0.373 |
| lazy | 0.373 |
| loudly | 0.490 |
| moon | 0.490 |

---

## Document 3

```text
fox quick fox clever animal
```

### Raw TF-IDF

| Word | TF | IDF | TF × IDF |
|------|---:|----:|---------:|
| animal | 1 | 1.6931 | 1.6931 |
| clever | 1 | 1.6931 | 1.6931 |
| fox | 2 | 1.2877 | 2.5754 |
| quick | 1 | 1.2877 | 1.2877 |

### L2 Normalization

$$
Norm=
\sqrt{
1.6931^2+
1.6931^2+
2.5754^2+
1.2877^2
}
$$

$$
=
\sqrt{
2.8666+
2.8666+
6.6330+
1.6581
}
$$

$$
=
\sqrt{14.0243}
$$

$$
=3.745
$$

Normalized values:

| Word | Value |
|------|------:|
| animal | 0.452 |
| clever | 0.452 |
| fox | 0.688 |
| quick | 0.344 |

---

# Step 5: Final TF-IDF Matrix

| Document | animal | bark | brown | clever | dog | fox | jump | lazy | loudly | moon | quick |
|-----------|:------:|:----:|:------:|:-------:|:---:|:---:|:----:|:----:|:------:|:----:|:-----:|
| D1 | 0.000 | 0.000 | 0.481 | 0.000 | 0.366 | 0.366 | 0.481 | 0.366 | 0.000 | 0.000 | 0.366 |
| D2 | 0.000 | 0.490 | 0.000 | 0.000 | 0.373 | 0.000 | 0.000 | 0.373 | 0.490 | 0.490 | 0.000 |
| D3 | 0.452 | 0.000 | 0.000 | 0.452 | 0.000 | 0.688 | 0.000 | 0.000 | 0.000 | 0.000 | 0.344 |

---

## Python Verification

```python
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

tfidf = TfidfVectorizer()

X = tfidf.fit_transform(corpus)

print(tfidf.get_feature_names_out())
print(np.round(X.toarray(), 3))
```

In [24]:
b=[" ".join(res) for res in results]
print(b)
cb=CountVectorizer()
x=cb.fit_transform(b)
print(cb.get_feature_names_out())
print(x.toarray())
tfidf=TfidfVectorizer()
y=tfidf.fit_transform(b)
print(tfidf.get_feature_names_out())
print(np.round(y.toarray(),3))

['quick brown fox jump lazy dog', 'lazy dog bark loudly moon', 'fox quick fox clever animal']
['animal' 'bark' 'brown' 'clever' 'dog' 'fox' 'jump' 'lazy' 'loudly'
 'moon' 'quick']
[[0 0 1 0 1 1 1 1 0 0 1]
 [0 1 0 0 1 0 0 1 1 1 0]
 [1 0 0 1 0 2 0 0 0 0 1]]
['animal' 'bark' 'brown' 'clever' 'dog' 'fox' 'jump' 'lazy' 'loudly'
 'moon' 'quick']
[[0.    0.    0.481 0.    0.366 0.366 0.481 0.366 0.    0.    0.366]
 [0.    0.49  0.    0.    0.373 0.    0.    0.373 0.49  0.49  0.   ]
 [0.452 0.    0.    0.452 0.    0.688 0.    0.    0.    0.    0.344]]


In [17]:
print(len(results))

3


## N-Gram Model Construction

The `ngram()` function builds an **N-gram language model**. For this experiment, **n = 2**, so it creates a **Bigram Model**. A bigram model estimates the probability of a word occurring given the previous word.

### Function Definition

```python
def ngram(data, n=2):
```

- `data` is a list of preprocessed tokens.
- `n=2` specifies that a **Bigram** model will be built.
- If `n=3`, the function builds a **Trigram** model.

Example token list:

```python
data = ['quick', 'brown', 'fox', 'jump']
```

---

### Step 1: Create a Dictionary for Frequency Counting

```python
m = defaultdict(Counter)
```

A dictionary is created where:

- **Key** = History (previous word)
- **Value** = Counter object storing frequencies of following words

Initially,

```python
m = {}
```

When a new history is encountered, `defaultdict` automatically creates an empty `Counter`.

Example:

```python
m[('quick',)]
```

becomes

```python
Counter()
```

---

### Step 2: Slide an N-word Window Through the Tokens

```python
for i in range(len(data)-n+1):
```

This loop moves an N-word window through the token sequence.

Suppose

```python
data = ['quick', 'brown', 'fox', 'jump']
n = 2
```

Then

$$
\text{Number of windows}
=
\text{len(data)}-n+1
$$

$$
=4-2+1=3
$$

Therefore,

```python
i = 0
i = 1
i = 2
```

---

### Step 3: Extract the History

```python
h = tuple(data[i:i+n-1])
```

For a Bigram,

$$
n-1=1
$$

Therefore, the history contains only one word.

For

```python
i = 0
```

```python
data[0:1]
```

returns

```python
['quick']
```

which is converted into

```python
('quick',)
```

A tuple is used because dictionary keys must be immutable.

---

### Step 4: Select the Next Word

```python
nw = data[i+n-1]
```

For

```python
i = 0
```

```python
nw = data[1]
```

Therefore,

```python
nw = 'brown'
```

The first training pair becomes

| History | Next Word |
|---------|-----------|
| ('quick',) | brown |

---

### Step 5: Count the Frequency

```python
m[h][nw] += 1
```

Initially,

```python
m = {}
```

After processing

```text
quick → brown
```

the dictionary becomes

```python
{
    ('quick',): Counter({'brown':1})
}
```

After processing

```text
brown → fox
```

```python
{
    ('quick',): Counter({'brown':1}),
    ('brown',): Counter({'fox':1})
}
```

Suppose another sentence contains

```text
quick → fox
```

then

```python
{
    ('quick',): Counter({
        'brown':1,
        'fox':1
    })
}
```

The history `"quick"` has now been followed by two different words.

---

### Step 6: Create a Probability Dictionary

```python
pm = defaultdict(dict)
```

A new dictionary is created to store probabilities instead of frequencies.

Initially,

```python
pm = {}
```

---

### Step 7: Process Every History

```python
for h, c in m.items():
```

Suppose

```python
m = {
    ('quick',): Counter({
        'brown':2,
        'fox':3
    })
}
```

Then

```python
h = ('quick',)
c = Counter({'fox':3,'brown':2})
```

---

### Step 8: Compute the Total Frequency

```python
total = sum(c.values())
```

Here,

```python
Counter({
    'brown':2,
    'fox':3
})
```

Total frequency is

$$
\text{Total}
=
2+3
=
5
$$

---

### Step 9: Compute the Probability of Each Next Word

```python
for nw, f in c.items():
```

Each next word is processed one at a time.

First iteration

```python
nw = 'brown'
f = 2
```

Second iteration

```python
nw = 'fox'
f = 3
```

---

### Step 10: Convert Frequency into Probability

```python
pm[h][nw] = f / total
```

The probability of a next word given a history is calculated using Maximum Likelihood Estimation (MLE):

$$
P(w_i|h)=
\frac{\text{Count}(h,w_i)}
{\text{Count}(h)}
$$

where

- $h$ = history (previous word)
- $w_i$ = next word

For the history **quick**

```text
brown : 2
fox   : 3
```

The probabilities become

$$
P(\text{brown}|\text{quick})
=
\frac{2}{5}
=
0.40
$$

$$
P(\text{fox}|\text{quick})
=
\frac{3}{5}
=
0.60
$$

Therefore,

```python
pm = {
    ('quick',):{
        'brown':0.40,
        'fox':0.60
    }
}
```

Notice that

$$
0.40+0.60=1.00
$$

which means the probabilities form a valid probability distribution.

---

### Step 11: Return the Language Model

```python
return pm
```

The function returns a dictionary containing all Bigram probabilities.

Example:

```python
{
    ('quick',):{
        'brown':0.50,
        'fox':0.50
    },
    ('brown',):{
        'fox':1.00
    },
    ('fox',):{
        'jump':0.50,
        'clever':0.50
    }
}
```

---

## Overall Algorithm

1. Read the token sequence.
2. Move an N-word sliding window across the sequence.
3. Split each window into:
   - **History** (first \(n-1\) words)
   - **Next Word** (last word)
4. Count how many times each next word follows the history.
5. Convert the counts into probabilities using

$$
P(w_i|h)=
\frac{\text{Count}(h,w_i)}
{\text{Count}(h)}
$$

6. Store the probabilities in a dictionary and return the completed N-gram language model.

In [25]:
def ngram(data, n=2):
    m=defaultdict(Counter)
    for i in range(len(data)-n+1):
        h=tuple(data[i:i+n-1])
        nw=data[i+n-1]
        m[h][nw]+=1
    pm=defaultdict(dict)
    for h,c in m.items():
        total=sum(c.values())
        for nw,f in c.items():
            pm[h][nw]=f/total   
    return pm
w=[]       
for r in results: 
    w.extend(r)

ans=ngram(w, n=2)

for h in ans:
    print(f"History {h}:")
    for nw, p in ans[h].items():
        print(f"  Next word '{nw}': {p:.2f}")

History ('quick',):
  Next word 'brown': 0.50
  Next word 'fox': 0.50
History ('brown',):
  Next word 'fox': 1.00
History ('fox',):
  Next word 'jump': 0.33
  Next word 'quick': 0.33
  Next word 'clever': 0.33
History ('jump',):
  Next word 'lazy': 1.00
History ('lazy',):
  Next word 'dog': 1.00
History ('dog',):
  Next word 'lazy': 0.50
  Next word 'bark': 0.50
History ('bark',):
  Next word 'loudly': 1.00
History ('loudly',):
  Next word 'moon': 1.00
History ('moon',):
  Next word 'fox': 1.00
History ('clever',):
  Next word 'animal': 1.00
